In [7]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from analysis.loaders import Session

pd.set_option("display.float_format", lambda v: f"{v:.3f}")

In [10]:
from pathlib import Path

# A session dir OR the .h5 path both work.
data_dir = Path("/Users/hakan/Library/CloudStorage/Dropbox/Hakan/lab/data/cheese_data/HK1/HK1_20260810/HK1_20260810_005/")
s = Session(data_dir)
s

Session('HK1_20260810_005', n_trials=24)

In [11]:
# Photodiode display-sync check. These are all built-in Session.trials columns:
#   latency_ms          = (true_onset_t - stim_onset_t) * 1e3   photodiode-measured display latency
#   first_pulse_lat_ms  = (first sync pulse - stim_onset_t) * 1e3
#   sync_ok             = 1 if the online onset-sync landed that trial
t = s.trials
print(f"{s.n_trials} trials | sync_ok {int(t['sync_ok'].sum())}/{len(t)} | "
      f"display latency median {t['latency_ms'].median():.1f} ms "
      f"[{t['latency_ms'].min():.1f}, {t['latency_ms'].max():.1f}]")
t[["stim_az_deg", "sync_ok", "latency_ms", "first_pulse_lat_ms"]]

24 trials | sync_ok 24/24 | display latency median 66.8 ms [55.8, 78.0]


,stim_az_deg,sync_ok,latency_ms,first_pulse_lat_ms
0,80.000,1,68.330,68.330
1,-40.000,1,75.599,75.599
2,40.000,1,68.293,68.293
3,-80.000,1,67.208,67.208
4,80.000,1,69.247,69.247
5,-40.000,1,68.715,68.715
6,40.000,1,62.148,62.148
7,-80.000,1,65.209,65.209
8,80.000,1,61.737,61.737
9,-40.000,1,73.415,73.415


In [12]:
# Corrected stim onset (true_onset_t) + reward delivery — also built-in Session.trials columns:
#   reward_lat_ms = (reward_t - true_onset_t) * 1e3        reward timing vs the corrected onset
#   rt_ms         = (first_lick_t - response_window_t)*1e3  first in-window lick vs window open
#   pavlovian     = True for auto/rescue reward, False for operant (delivered at the lick)
cols = ["stim_az_deg", "trial_outcome", "sync_ok", "latency_ms",
        "reward_ul", "pavlovian", "reward_lat_ms", "rt_ms"]
s.trials[cols]

,stim_az_deg,trial_outcome,sync_ok,latency_ms,reward_ul,pavlovian,reward_lat_ms,rt_ms
0,80.000,miss,1,68.330,4.000,False,1.925,NaN
1,-40.000,miss,1,75.599,4.000,False,1.436,NaN
2,40.000,miss,1,68.293,4.000,False,1.199,NaN
3,-80.000,miss,1,67.208,4.000,False,1.601,NaN
4,80.000,miss,1,69.247,4.000,False,2.158,NaN
5,-40.000,miss,1,68.715,4.000,False,1.010,NaN
6,40.000,miss,1,62.148,4.000,False,2.092,NaN
7,-80.000,miss,1,65.209,4.000,False,0.740,NaN
8,80.000,miss,1,61.737,4.000,False,1.348,NaN
9,-40.000,miss,1,73.415,4.000,False,1.281,NaN


In [13]:
# Raw per-trial sync pulses live in Session.pulses (one row per rising edge, t_rel_onset_ms =
# pulse time - stim_onset_t). The first pulse of each trial is the display-onset marker used
# for latency above; count + first pulse per trial:
s.pulses.groupby("trial_num")["t_rel_onset_ms"].agg(n_pulses="count", first_pulse_ms="first")

,n_pulses,first_pulse_ms
trial_num,,
0,49,68.330
1,46,75.599
2,44,68.293
3,46,67.208
4,50,69.247
5,48,68.715
6,52,62.148
7,46,65.209
8,48,61.737
